# 

In [ ]:
import numpy as np
import adi
import matplotlib.pyplot as plt
import subprocess
from time import sleep
                



mem_map = {"i_ampl":           0x43C00000,
            "q_ampl":           0x43C00004,
            "frequency":        0x43C0000C,
            "multiplier":       0x43C00014,
            "Phase_PDH":        0x43C00010,
            "phase_difference": 0x43C00008,
            "i_q_offsets": 0x43C00018 }

sample_rate = 61.44e6 # Hz


subprocess.call("sshpass -p 'analog' ssh root@192.168.2.1 'echo 954  > /sys/class/gpio/export'",shell=True)
subprocess.call("sshpass -p 'analog' ssh root@192.168.2.1 'echo out >  /sys/class/gpio/gpio954/direction'",shell=True)


def external_clock():
    subprocess.call("sshpass -p 'analog' ssh root@192.168.2.1 'echo 1 >  /sys/class/gpio/gpio954/value'",shell=True)
    
def internal_clock():
    subprocess.call("sshpass -p 'analog' ssh root@192.168.2.1 'echo 0 >  /sys/class/gpio/gpio954/value'",shell=True)
    

def change_I_Q_ampl(i,q):
    subprocess.call(f"sshpass -p 'analog' ssh root@192.168.2.1 'devmem {mem_map['i_ampl']} 32 {i}'",shell=True)
    subprocess.call(f"sshpass -p 'analog' ssh root@192.168.2.1 'devmem {mem_map['q_ampl']} 32 {q}'",shell=True)
    
def change_phase_diff(p):
    subprocess.call(f"sshpass -p 'analog' ssh root@192.168.2.1 'devmem {mem_map['phase_difference']} 32 {p}'",shell=True)
    
def change_phase_PDH(pdh):
    subprocess.call(f"sshpass -p 'analog' ssh root@192.168.2.1 'devmem {mem_map['Phase_PDH']} 32 {pdh}'",shell=True)

    
def change_frequency(frequency):
    sdr.tx_lo = frequency
    
def change_mod_frequency(frequency, sampling_rate = sample_rate):    
    frequency_code = int(frequency * 0xFFFF_FFFF / sampling_rate / 8) # divided by 8 because the PDH generator runs 8 times faster
    subprocess.call(f"sshpass -p 'analog' ssh root@192.168.2.1 'devmem {mem_map['frequency']} 32 {frequency_code}'",shell=True)
    print(frequency_code)
    
def change_mod_depth(depth=42):
    subprocess.call(f"sshpass -p 'analog' ssh root@192.168.2.1 'devmem {mem_map['multiplier']} 32 {depth}'",shell=True)
    
    
def change_i_q_offsets(I=0,Q=0):
    i = np.array(I,dtype=np.int16).tobytes()
    q = np.array(Q,dtype=np.int16).tobytes()
    out = i + q
    out = int(np.frombuffer(out,dtype=np.uint32))
    subprocess.call(f"sshpass -p 'analog' ssh root@192.168.2.1 'devmem {mem_map['i_q_offsets']} 32 {out}'",shell=True)
    

def load_settings(settings):
    for key,value in settings.items():
        subprocess.call(f"sshpass -p 'analog' ssh root@192.168.2.1 'devmem {value[0]} 32 {value[1]}'",shell=True)
    
    
# Configuring the FPGA    
subprocess.call("sshpass -p 'analog' scp system_top.bit.bin root@192.168.2.1:/lib/firmware/.",shell=True)
subprocess.call("sshpass -p 'analog' scp configure_FPGA.sh  root@192.168.2.1:/root",shell=True)               
subprocess.call("sshpass -p 'analog' ssh root@192.168.2.1 'sh configure_FPGA.sh'",shell=True)


sdr = adi.Pluto("ip:192.168.2.1")
sdr.sample_rate = int(sample_rate)


# Config Tx
sdr.tx_rf_bandwidth = int(sample_rate) # filter cutoff, just set it to the same as sample rate

sdr.tx_hardwaregain_chan0 = 0 # Increase to increase tx power, valid range is -90 to 0 dB

# Start the transmitter
i = np.zeros(16) #Generating "fake" samples to enable transmitter
q = np.zeros(16)
samples= i+1j*q#
sdr.tx_cyclic_buffer = True # Enable cyclic buffers
sdr.tx(samples) # start transmitting


external_clock()

change_phase_diff(0x00100000)

In [ ]:
change_frequency(1_500_000_000)

change_i_q_offsets(I=0,Q=0)

change_I_Q_ampl(0x7FFF,0x7FFF)

change_phase_diff(int(0x00100000 * 0.995 ) )


change_mod_frequency(3_072_000)



change_mod_depth(80)